# 04 — WBIC Model Fitting

Fits RL models per session using the Watanabe-Bayesian Information Criterion (WBIC).

**Models**
| Key | Description | Parameters |
|-----|-------------|------------|
| `mf_p`      | Model-free + perseveration           | alpha, forget, lambda, Wmf, P |
| `mb_p`      | Model-based + perseveration          | alpha, forget, Wmb, P |
| `rac_p`     | Reward-as-cue + perseveration        | alpha, tmp, P |
| `hyb_p`     | Hybrid (MF+MB) + perseveration       | alpha, forget, lambda, Wmf, Wmb, P |
| `ls`        | Latent State (Akam 2015)             | p_r, beta |
| `ls_asym_p` | Latent State Asymmetric + P          | p_r, i_temp, P |
| `hyb_inf`   | Hybrid (Asym Inference + MF) + P     | alpha, forget, lambda, Wmf, Winf, P, p_r |

**Outputs** (per model, per stage):
- `results/WBIC/WBIC_{model}_{stage}.csv`  — one row per session
- `results/WBIC/WBIC_subj_{model}_{stage}.csv` — per-subject mean ± SD

**Tip**: set `MODELS_TO_FIT` to a single model and run one at a time to manage runtime.

In [1]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────

# MODELS_TO_FIT = ['mf_p', 'mb_p', 'rac_p', 'hyb_p', 'ls', 'ls_asym_p', 'hyb_inf']

MODELS_TO_FIT = ['ls']
STAGES = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']

# Stan sampling settings (matches 02k reference notebook)
N_ITER   = 5000
N_WARMUP = 750
N_CHAINS = 4
SEED     = 123

# Set True to refit even if the output CSV already exists
FORCE_REFIT = True

In [2]:
# ── Cell 2: Imports & path setup ──────────────────────────────────────────

import sys, os

# Locate organized/ regardless of where the notebook is launched from.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    _organized = os.path.dirname(_here)
else:
    _organized = os.path.join(_here, 'organized')
    if not os.path.isdir(_organized):
        _organized = _here
sys.path.insert(0, _organized)
# Also add src/ so that pickle can reconstruct Session objects whose
# class was saved as 'data_import.Session' (not 'src.data_import.Session').
sys.path.insert(0, os.path.join(_organized, 'src'))

import numpy as np
import pandas as pd

from config import (
    SUBJECTS, RAW_DATA_DIR,
    raw_subject_dir, wbic_result_csv, wbic_subj_csv
)
from src.data_import import Experiment
from src.fitting import RL_data_arrange_single
from src.wbic import WBIC_MODELS, compile_wbic_models, fit_wbic_session

print('Organized root:', _organized)
print('Subjects:', SUBJECTS)

Organized root: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison
Subjects: ['WT1', 'WT2', 'WT3', 'WT4', 'WT5', 'WT6']


In [3]:
# ── Cell 3: Session loading helper ────────────────────────────────────────
#
# Stage mapping (mirrors 02_model_fitting.ipynb):
#   '4.8'        → main folder  (WT_data/WT1/),          ALL sessions, no filter
#   '4.2'–'4.7'  → Training folder (WT_data/WT1_Training/), filtered by training_stage
#
# Note: sessions in the main folder have training_stage=='4.7' in the .txt files,
# but the analysis labels them '4.8' (final experiment sessions vs. training sessions).

def load_sessions(subject, stage):
    if stage == '4.8':
        # Final experiment sessions live in the main (non-Training) folder.
        # No training_stage filter — return all sessions in that folder.
        folder = raw_subject_dir(subject, training=False)
        if not os.path.isdir(folder):
            return []
        exp = Experiment(folder)
        sessions = list(exp.sessions)
    else:
        # Early/mid training stages live in the _Training folder.
        folder = raw_subject_dir(subject, training=True)
        if not os.path.isdir(folder):
            return []
        exp = Experiment(folder)
        sessions = [s for s in exp.sessions if s.training_stage == stage]

    # Sort chronologically
    return sorted(sessions, key=lambda s: s.datetime_string)


# Quick check: show session counts for each subject × stage
print(f"{'Subject':<10}", end='')
for stage in STAGES:
    print(f"{stage:>6}", end='')
print()
print('-' * (10 + 6 * len(STAGES)))

for subject in SUBJECTS:
    print(f"{subject:<10}", end='')
    for stage in STAGES:
        n = len(load_sessions(subject, stage))
        print(f"{n:>6}", end='')
    print()

Subject      4.2   4.3   4.4   4.5   4.6   4.7   4.8
----------------------------------------------------
WT1       Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     5Saved sessions loaded from: sessions.pkl
    10Saved sessions loaded from: sessions.pkl
    11
WT2       Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
    11Saved sessions loaded from: sessions.pkl
     2Saved sessions loaded from: sessions.pkl
    17
WT3       Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions loaded from: sessions.pkl
     1Saved sessions l

In [4]:
# ── Cell 4: Compile Stan models (slow, ~30 s each) ─────────────────────────

# Only compile models that have at least one (model, stage) pair to fit.
models_needed = []
for model_key in MODELS_TO_FIT:
    for stage in STAGES:
        csv_path = wbic_result_csv(model_key, stage)
        if FORCE_REFIT or not os.path.exists(csv_path):
            if model_key not in models_needed:
                models_needed.append(model_key)
            break

print(f'Models to compile: {models_needed}')
compiled = compile_wbic_models(models_needed)
print('All models compiled.')

Models to compile: ['ls']


Compiling Stan model: ls ...


INFO:pystan:COMPILING THE C++ CODE FOR MODEL anon_model_86617b852eddce6cbefa5379c8776261 NOW.
In file included from /gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarraytypes.h:1940,
                 from /gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from /tmp/pystan_m9squ96d/stanfit4anon_model_86617b852eddce6cbefa5379c8776261_3947402780217453311.cpp:1315:
/gpfs/radev/home/my483/.conda/envs/stanenv/lib/python3.8/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
In file included from /tmp/pystan_

  done.
All models compiled.


In [5]:
# ── Cell 5: Main fitting loop ──────────────────────────────────────────────
#
# For each (model, stage): fit every session of every subject, then save CSV.
# Progress is printed at each (model, stage, subject) level.

for model_key in MODELS_TO_FIT:
    if model_key not in compiled:
        print(f'[{model_key}] already complete for all stages — skipping.')
        continue

    sm         = compiled[model_key]
    model_info = WBIC_MODELS[model_key]

    for stage in STAGES:
        csv_path = wbic_result_csv(model_key, stage)

        if not FORCE_REFIT and os.path.exists(csv_path):
            print(f'[{model_key} | stage {stage}] CSV exists — skipping.')
            continue

        print(f'\n=== [{model_key} | stage {stage}] ===')
        rows = []

        for subject in SUBJECTS:
            sessions = load_sessions(subject, stage)
            if not sessions:
                print(f'  {subject}: no sessions found for stage {stage}')
                continue

            print(f'  {subject}: {len(sessions)} sessions', flush=True)

            for sess_idx, session in enumerate(sessions):
                print(f'    session {sess_idx} ({session.datetime_string}) ...', flush=True)

                try:
                    T, c, ss, tt, r, PR, PL = RL_data_arrange_single(session)

                    result = fit_wbic_session(
                        sm, model_info, T, c, ss, tt, r, PR, PL,
                        n_iter=N_ITER, n_warmup=N_WARMUP,
                        n_chains=N_CHAINS, seed=SEED
                    )

                    row = {
                        'subject':     subject,
                        'session_idx': sess_idx,
                        'date':        session.datetime_string,
                        'stage':       stage,
                        'model':       model_key,
                    }
                    row.update(result)   # adds WBIC, n_trials, and all params
                    rows.append(row)

                    print(f'      WBIC = {result["WBIC"]:.2f}', flush=True)

                except Exception as e:
                    print(f'      ERROR: {e}')

        if rows:
            df = pd.DataFrame(rows)
            df.to_csv(csv_path, index=False)
            print(f'  Saved: {csv_path}')
        else:
            print(f'  No data collected for [{model_key} | stage {stage}].')

print('\nFitting complete.')


=== [ls | stage 4.2] ===
Saved sessions loaded from: sessions.pkl
  WT1: 1 sessions
    session 0 (2019-12-18 16:00:11) ...



Gradient evaluation took 7.4e-05 secondsGradient evaluation took 7.2e-05 seconds

Gradient evaluation took 7.3e-05 seconds
1000 transitions using 10 leapfrog steps per transition would take 0.74 seconds.
Adjust your expectations accordingly!1000 transitions using 10 leapfrog steps per transition would take 0.72 seconds.
1000 transitions using 10 leapfrog steps per transition would take 0.73 seconds.


Adjust your expectations accordingly!Adjust your expectations accordingly!







Gradient evaluation took 5.9e-05 seconds
1000 transitions using 10 leapfrog steps per transition would take 0.59 seconds.
Adjust your expectations accordingly!


Iteration:    1 / 5000 [  0%]  (Warmup)
Iteration:    1 / 5000 [  0%]  (Warmup)
Iteration:    1 / 5000 [  0%]  (Warmup)
Iteration:    1 / 5000 [  0%]  (Warmup)
Iteration:  500 / 5000 [ 10%]  (Warmup)
Iteration:  500 / 5000 

In [6]:
# ── Cell 6: Per-subject summary ────────────────────────────────────────────
#
# For each (model, stage) CSV that exists, compute per-subject mean ± SD
# of WBIC and all parameters across sessions, then save a summary CSV.

for model_key in MODELS_TO_FIT:
    param_names = WBIC_MODELS[model_key].param_names
    value_cols  = ['WBIC'] + param_names

    for stage in STAGES:
        csv_path  = wbic_result_csv(model_key, stage)
        subj_path = wbic_subj_csv(model_key, stage)

        if not os.path.exists(csv_path):
            continue

        df = pd.read_csv(csv_path)

        # Aggregate per subject: mean and SD across sessions
        agg = df.groupby('subject')[value_cols].agg(['mean', 'std']).reset_index()
        agg.columns = ['subject'] + [
            f'{col}_{stat}'
            for col in value_cols
            for stat in ('mean', 'std')
        ]
        agg.insert(1, 'n_sessions', df.groupby('subject').size().values)

        agg.to_csv(subj_path, index=False)
        print(f'Saved summary: {subj_path}')

print('Summaries complete.')

Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.2.csv
Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.3.csv
Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.4.csv
Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.5.csv
Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.6.csv
Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.7.csv
Saved summary: /gpfs/radev/project/kuan/my483/RL_twostep/twostep-model-comparison/results/WBIC/WBIC_subj_ls_4.8.csv
Summaries complete.


In [7]:
# ── Cell 7: Progress table ─────────────────────────────────────────────────
#
# Shows which (model × stage) CSV files exist vs. are missing.

print(f"{'Model':<12}", end='')
for stage in STAGES:
    print(f"{stage:>6}", end='')
print()
print('-' * (12 + 6 * len(STAGES)))

for model_key in MODELS_TO_FIT:
    print(f"{model_key:<12}", end='')
    for stage in STAGES:
        csv_path = wbic_result_csv(model_key, stage)
        symbol = ' OK ' if os.path.exists(csv_path) else ' -- '
        print(f"{symbol:>6}", end='')
    print()

Model          4.2   4.3   4.4   4.5   4.6   4.7   4.8
------------------------------------------------------
ls             OK    OK    OK    OK    OK    OK    OK 
